# 02. 蔵元の所在地・水道水の硬度データの収集

日本酒の味わいと「水」の関係を調べるための補助データを2種類収集します。

| 収集対象 | 取得元 | 出力 |
|---|---|---|
| 蔵元の所在地（住所） | Saketimeの銘柄ページ（Playwright） | `maker_address_master.csv` |
| 全国の浄水場ごとの水道水の硬度・pH | 水道水データ一覧ページ（requests + BeautifulSoup） | `national_water_hardness_master.csv` |

- 蔵元名は `01_scrape_reviews.ipynb` と同じロジックで取得し、表記揺れが出ないようにしています
- 蔵元住所は県ごとに中間保存（途中再開可）し、最後に蔵元単位で重複を排除して統合します

In [ ]:
# Playwright（ヘッドレスChromium）のセットアップ
!pip install -q playwright
!playwright install chromium

# Chromiumの実行に必要なシステムライブラリ
!apt-get update -qq
!apt-get install -y -qq libatk-adaptor libatk-bridge2.0-0 libxcomposite1

## 設定

In [ ]:
# --- 実行環境の設定 ---------------------------------------------------------
# Colab: Google Driveをマウントし、BASE_DIR 配下を読み書きします。
# ローカル: 環境変数 SAKE_PROJECT_DIR でデータ置き場を指定できます。
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析'  # 自分のDriveのパスに変更
except ImportError:
    BASE_DIR = os.environ.get('SAKE_PROJECT_DIR', './sake_review_analysis')

CSV_DIR = f'{BASE_DIR}/口コミ_CSV'          # 収集・加工したCSV
for d in (CSV_DIR,):
    os.makedirs(d, exist_ok=True)

## 蔵元の所在地マスタ

In [ ]:
import asyncio
import re
import random
import os
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd

MASTER_OUTPUT_PATH = f'{CSV_DIR}/maker_address_master.csv'

async def fetch_brand_urls_by_prefecture(page, pref_code, limit=10):
    """
    【処理1】各都道府県のランキングから上位の銘柄URLを自動収集（前回と同一）
    """
    url = f"https://www.saketime.jp/ranking/{pref_code}/"
    print(f"🌐 巡回ルート作成中... [{pref_code.upper()}] ランキングから上位 {limit} 件の銘柄を抽出します。")
    try:
        await page.goto(url, timeout=60000, wait_until='domcontentloaded')
        await asyncio.sleep(3)
        html = await page.content()
        soup = BeautifulSoup(html, 'html.parser')
        brand_urls = []
        for a_tag in soup.find_all('a', href=re.compile(r'^/brands/\d+/$')):
            full_url = f"https://www.saketime.jp{a_tag['href']}"
            if full_url not in brand_urls:
                brand_urls.append(full_url)
            if len(brand_urls) >= limit:
                break
        print(f"  ➔ [{pref_code.upper()}] の巡回対象: {len(brand_urls)} 銘柄が確定しました。")
        return brand_urls
    except Exception as e:
        print(f"  ⚠️ [{pref_code.upper()}] ランキングの自動取得に失敗しました: {e}")
        return []

async def scrape_brand_address(page, brand_url):
    """
    【処理2】銘柄ページから「蔵元名」と「所在地（住所）」をピンポイントで抽出する関数
    """
    try:
        await page.goto(brand_url, timeout=60000, wait_until='domcontentloaded')
        await asyncio.sleep(3) # レンダリング待機

        html_content = await page.content()
        soup = BeautifulSoup(html_content, 'html.parser')

        # 1. 蔵元名の抽出（前回の口コミ収集コードと完全同一ロジック ➔ 表記揺れが絶対に起きない）
        maker_info = "不明"
        maker_element = soup.find('a', href=re.compile(r'#maker$'))
        if not maker_element:
            brandinfo_p = soup.find('p', class_='brandinfo')
            if brandinfo_p:
                a_tags = brandinfo_p.find_all('a')
                if len(a_tags) >= 2:
                    maker_element = a_tags[1]
        if maker_element:
            maker_info = maker_element.get_text(strip=True)

        # 2. 所在地の抽出（ユーザーが特定した th/td 構造から確実に取得）
        address_info = "不明"
        for th in soup.find_all('th'):
            if '所在地' in th.get_text():
                td = th.find_next_sibling('td')
                if td:
                    address_info = td.get_text(strip=True)
                    break

        return {
            "maker": maker_info,
            "address": address_info
        }
    except Exception as e:
        print(f"  ❌ ページ取得エラー ({brand_url}): {e}")
        return None

async def main_address_master_crawler(pref_list, max_brands_per_pref=10):
    """
    【メイン管制塔】県別中間保存・自動再開（レジューム）機能付き
    """
    output_dir = os.path.dirname(MASTER_OUTPUT_PATH)
    os.makedirs(output_dir, exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            viewport={'width': 1280, 'height': 800}
        )
        page = await context.new_page()

        for pref in pref_list:
            # 県別の住所一時保存ファイル
            pref_output_path = os.path.join(output_dir, f"tmp_address_{pref}.csv")

            print(f"\n==================================================")
            print(f"🗺️ 【住所抽出】都道府県「{pref.upper()}」の調査")
            print(f"==================================================")

            # すでに該当県の一時ファイルがあれば自動スキップ
            if os.path.exists(pref_output_path):
                print(f"⏭️ [{pref.upper()}] は既に住所取得済みのためスキップします。")
                continue

            target_urls = await fetch_brand_urls_by_prefecture(page, pref_code=pref, limit=max_brands_per_pref)
            pref_rows = []

            for index, url in enumerate(target_urls, start=1):
                print(f" ➔ [{index} / {len(target_urls)}] 住所抽出中... ({url})")
                result = await scrape_brand_address(page, url)

                if result and result['maker'] != "不明":
                    pref_rows.append({
                        "prefecture": pref,
                        "maker": result['maker'],
                        "address": result['address']
                    })
                    print(f"    ✨ 特定: {result['maker']} ➔ {result['address']}")

                # サーバー負荷軽減のための休憩（住所だけなので5〜8秒に調整）
                if index < len(target_urls):
                    wait_time = random.uniform(5.0, 8.0)
                    await asyncio.sleep(wait_time)

            # 県ごとにCSVへ書き出し
            if pref_rows:
                df_pref = pd.DataFrame(pref_rows)
                df_pref.to_csv(pref_output_path, index=False, encoding='utf-8-sig')
                print(f"📂 [{pref.upper()}] の住所データを中間保存しました。")

            await asyncio.sleep(10) # 県の切り替え時に10秒休憩

        await browser.close()

    # =========================================================
    # 🔗 最終フェーズ：分割された住所CSVを1つに統合＆重複排除
    # =========================================================
    print("\n🔗 ==================================================")
    print("🔗 全都道府県の住所データの統合処理を開始します...")
    print("🔗 ==================================================")

    combined_dfs = []
    for pref in pref_list:
        pref_output_path = os.path.join(output_dir, f"tmp_address_{pref}.csv")
        if os.path.exists(pref_output_path):
            combined_dfs.append(pd.read_csv(pref_output_path))

    if combined_dfs:
        df_final_master = pd.concat(combined_dfs, ignore_index=True)

        # 💡【重要】同じ蔵元が複数銘柄を展開している場合の重複を排除（一意なマスタにする）
        df_final_master = df_final_master.drop_duplicates(subset=['prefecture', 'maker'], keep='first')

        df_final_master.to_csv(MASTER_OUTPUT_PATH, index=False, encoding='utf-8-sig')
        print(f"\n🎉 住所マスタが完成しました！\n➡️ {MASTER_OUTPUT_PATH}")
        print(f"📊 総登録蔵元数: {len(df_final_master)} 蔵")
    else:
        print("⚠️ 結合対象のデータが見つかりませんでした。")

# 対象44府県リスト
target_44_prefectures = [
    'hokkaido', 'aomori', 'iwate', 'miyagi', 'akita', 'yamagata', 'fukushima',
    'ibaraki', 'tochigi', 'gunma', 'saitama', 'chiba', 'tokyo', 'kanagawa',
    'niigata', 'toyama', 'ishikawa', 'fukui', 'yamanashi', 'nagano', 'gifu',
    'shizuoka', 'aichi', 'mie', 'shiga', 'kyoto', 'osaka', 'hyogo', 'nara',
    'wakayama', 'tottori', 'shimane', 'okayama', 'hiroshima', 'yamaguchi',
    'tokushima', 'kagawa', 'ehime', 'kochi', 'fukuoka', 'saga', 'nagasaki',
    'kumamoto', 'oita'
]

# クローラー起動
await main_address_master_crawler(pref_list=target_44_prefectures, max_brands_per_pref=10)

## 全国水道水硬度データベース

In [ ]:
import os
import time
import random
import requests
from bs4 import BeautifulSoup
import pandas as pd

# ⚙️ 設定（保存先パスを環境に合わせて変更してください）
OUTPUT_HARDNESS_CSV = f'{CSV_DIR}/national_water_hardness_master.csv'

def get_prefecture_links():
    """
    【ステップ1】トップページから47都道府県のデータページのURLを自動収集する
    """
    top_url = "https://www.gex-fp.co.jp/fish/megreen/?page_id=1597"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    print("🌐 水道水データ一覧トップページを解析中...")
    res = requests.get(top_url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')

    pref_links = {}
    prefectures = [
        "北海道", "青森県", "岩手県", "宮城県", "秋田県", "山形県", "福島県",
        "茨城県", "栃木県", "群馬県", "埼玉県", "千葉県", "東京都", "神奈川県",
        "新潟県", "富山県", "石川県", "福井県", "山梨県", "長野県", "岐阜県",
        "静岡県", "愛知県", "三重県", "滋賀県", "京都府", "大阪府", "兵庫県",
        "奈良県", "和歌山県", "鳥取県", "島根県", "岡山県", "広島県", "山口県",
        "徳島県", "香川県", "愛媛県", "高知県", "福岡県", "佐賀県", "長崎県",
        "熊本県", "大分県", "宮崎県", "鹿児島県", "沖縄県"
    ]

    for a_tag in soup.find_all('a'):
        text = a_tag.get_text(strip=True)
        href = a_tag.get('href', '')

        if text in prefectures and 'page_id=' in href:
            pref_links[text] = href

    print(f"✨ 47都道府県中、 {len(pref_links)} 件のデータページURLを特定しました。")
    return pref_links

def scrape_hardness_table(pref_name, url):
    """
    【ステップ2】BeautifulSoupで<tr>と<td>を走査し、4つのカラムに正確に格納する
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    try:
        res = requests.get(url, headers=headers)
        # 文字化け防止対策
        res.encoding = res.apparent_encoding
        soup = BeautifulSoup(res.text, 'html.parser')

        table = soup.find('table')
        if not table:
            print(f"  ⚠️ {pref_name} のページにテーブルが見つかりません。")
            return None

        extracted_rows = []

        # テーブル内のすべての行(tr)をループ処理
        for tr in table.find_all('tr'):
            # 行内のすべてのセル(tdまたはth)を取得
            tds = tr.find_all(['td', 'th'])

            # データが不足している行（空行など）はスキップ
            if len(tds) < 4:
                continue

            # 各セルのテキストを綺麗にして抽出
            col_location = tds[0].get_text(strip=True) # 所在地
            col_plant    = tds[1].get_text(strip=True) # 浄水場名
            col_hardness = tds[2].get_text(strip=True) # 総硬度
            col_ph       = tds[3].get_text(strip=True) # pH

            # 🔥 ヘッダー行（「所在地」という文字そのものが入っている行）はデータではないのでスキップ
            if "所在地" in col_location:
                continue

            # 正常なデータ行であればリストに追加
            extracted_rows.append({
                '都道府県': pref_name,
                '所在地': col_location,
                '浄水場名': col_plant,
                '総硬度': col_hardness,
                'pH': col_ph
            })

        if not extracted_rows:
            return None

        # 最後にDataFrameに変換して返す
        return pd.DataFrame(extracted_rows)

    except Exception as e:
        print(f"  ❌ {pref_name} の解析中にエラーが発生しました: {e}")
        return None

def main():
    os.makedirs(os.path.dirname(OUTPUT_HARDNESS_CSV), exist_ok=True)
    pref_links = get_prefecture_links()

    if not pref_links:
        print("❌ 都道府県のリンクを取得できませんでした。処理を中断します。")
        return

    all_data = []

    for index, (pref_name, url) in enumerate(pref_links.items(), start=1):
        print(f"🚀 [{index}/47] データを解析中: {pref_name} ...")

        df_pref = scrape_hardness_table(pref_name, url)

        if df_pref is not None and not df_pref.empty:
            all_data.append(df_pref)
            print(f"  ➔ ✨ {len(df_pref)} 件の浄水場データをパースしました。")
        else:
            print(f"  ⚠️ {pref_name} からデータを抽出できませんでした。")

        # 相手サーバーへの負荷軽減（3〜5秒のランダム待機）
        time.sleep(random.uniform(3.0, 5.0))

    if all_data:
        df_national = pd.concat(all_data, ignore_index=True)

        # CSVファイルとして保存
        df_national.to_csv(OUTPUT_HARDNESS_CSV, index=False, encoding='utf-8-sig')

        print("\n==================================================")
        print(f"🎉 日本全国水道水硬度データベースが完成しました！")
        print(f"📁 保存先: {OUTPUT_HARDNESS_CSV}")
        print(f"📊 総データ数: {len(df_national)} 件")
        print("==================================================")
    else:
        print("❌ 結合できるデータがありませんでした。")

# 実行
main()

🌐 水道水データ一覧トップページを解析中...
✨ 47都道府県中、 47 件のデータページURLを特定しました。
🚀 [1/47] データを解析中: 北海道 ...
  ➔ ✨ 210 件の浄水場データをパースしました。
🚀 [2/47] データを解析中: 青森県 ...
  ➔ ✨ 129 件の浄水場データをパースしました。
🚀 [3/47] データを解析中: 岩手県 ...
  ➔ ✨ 276 件の浄水場データをパースしました。
🚀 [4/47] データを解析中: 宮城県 ...
  ➔ ✨ 151 件の浄水場データをパースしました。
🚀 [5/47] データを解析中: 秋田県 ...
  ➔ ✨ 158 件の浄水場データをパースしました。
🚀 [6/47] データを解析中: 山形県 ...
  ➔ ✨ 125 件の浄水場データをパースしました。
🚀 [7/47] データを解析中: 福島県 ...
  ➔ ✨ 193 件の浄水場データをパースしました。
🚀 [8/47] データを解析中: 茨城県 ...
  ➔ ✨ 165 件の浄水場データをパースしました。
🚀 [9/47] データを解析中: 栃木県 ...
  ➔ ✨ 219 件の浄水場データをパースしました。
🚀 [10/47] データを解析中: 群馬県 ...
  ➔ ✨ 245 件の浄水場データをパースしました。
🚀 [11/47] データを解析中: 埼玉県 ...
  ➔ ✨ 221 件の浄水場データをパースしました。
🚀 [12/47] データを解析中: 東京都 ...
  ➔ ✨ 29 件の浄水場データをパースしました。
🚀 [13/47] データを解析中: 千葉県 ...
  ➔ ✨ 182 件の浄水場データをパースしました。
🚀 [14/47] データを解析中: 神奈川県 ...
  ➔ ✨ 109 件の浄水場データをパースしました。
🚀 [15/47] データを解析中: 長野県 ...
  ➔ ✨ 380 件の浄水場データをパースしました。
🚀 [16/47] データを解析中: 山梨県 ...
  ➔ ✨ 191 件の浄水場データをパースしました。
🚀 [17/47] データを解析中: 新潟県 ...
  ➔ ✨ 298 件の浄水場データをパースしました。
🚀 [18/47] データを解析